# Create Garfield Weston Foundation Grant Awards (ORG-LEVEL GRANT PATTERN, Method 1 360Giving open data)

The Garfield Weston Foundation (GWF) is one of the largest UK grant-making charities (founded 1958), funding welfare, faith, community, youth, arts, health, education and environment work across the UK — including some research. This ingest covers the foundation's complete published grant record, one row per grant.

**Method 1 (direct open-data download).** GWF is a [360Giving](https://www.threesixtygiving.org/) publisher (registry prefix `360G-Weston`). The scraper (`scripts/local/garfield_weston_to_s3.py`) downloads the full grant corpus directly (a Google Sheets export, CC BY 4.0) in the 360Giving standard, using the project's reusable column-variant-tolerant 360Giving resolver. No browser automation.

**Awarding body:** Garfield Weston Foundation - F4320314718 (GB, DOI 10.13039/100013999). **NB:** this is the UK foundation — *not* F4320319983, which is the distinct Canadian "W. Garfield Weston Foundation".

**Source:** the 360Giving grants sheet. This is an **org-level grant funder** — each grant is made to a recipient **organization** (no named PI). Each grant carries a stable `Identifier`, a `Title`, a `Description`, an `Amount Awarded` (GBP), a real `Award date`, the `Recipient Org: Name`, and a `Grant Programme:Title`.

**Schema choices:**
- One row per grant `Identifier` (e.g. `360G-GWF-101945_2024-09-09`). The source re-exports the same Identifier across `Last modified` snapshots, so the scraper keeps the **latest snapshot per Identifier** (17,960 source rows → 15,504 unique grants), preventing double-counted amounts.
- `display_name` = the grant's own `Title`. `funding_type` = `grant`. `funder_scheme` = the `Grant Programme` (Welfare, Faith, Community, Youth, Arts, Health, Education, Environment).
- **Amount** = `Amount Awarded` in GBP → `amount` (DOUBLE) + `currency` = `GBP` (99.9% positive; the few 0/blank → NULL). **§6.7 NOT waived**, never imputed.
- `start_date` = the real `Award date` (true `YYYY-MM-DD`); `start_year` derived. `end_date`/`end_year` NULL (the file has no project end date — no false precision).
- `lead_investigator` = an **org-only** lead: `given_name`/`family_name` NULL, `affiliation.name` = the recipient org. **`affiliation.country` is NULL** — the GWF file has no recipient-country column (only an internal UK Beneficiary-Location code), so country is never guessed.
- `co_lead_investigator` and `investigators` are NULL. `description` = the grant's `Description`.

**Coverage note:** this is GWF's complete grant record (2017-2026); most grants fund UK welfare/faith/community work (Health/Education/Environment are the research-adjacent programmes). The funder is in OpenAlex (currently ~405 acknowledgement-linked works); the full record carries the authoritative `Identifier` for downstream matching.

**S3 location:** `s3a://openalex-ingest/awards/garfield_weston/garfield_weston_grants.parquet`

**Provenance:** `garfield_weston_foundation` (verified count=0 on production 2026-06-01).


## Step 1: Create staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.garfield_weston_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/garfield_weston/garfield_weston_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.garfield_weston_raw;


## Step 1.5: Inspect raw + programme/year/coverage

Expected: 15,504 unique grants (deduped to the latest snapshot per Identifier), 2017-2026. title / funder_award_id / start_date / start_year / recipient_org / grant_programme 100%; amount 99.9%; description ~100%.


In [ ]:
%sql
DESCRIBE openalex.awards.garfield_weston_raw;


In [ ]:
%sql
SELECT * FROM openalex.awards.garfield_weston_raw LIMIT 5;


In [ ]:
%sql
-- Per-(programme, start_year) row counts + amount coverage.
SELECT grant_programme, start_year, COUNT(*) AS rows,
       COUNT(amount) AS has_amount
FROM openalex.awards.garfield_weston_raw
GROUP BY grant_programme, start_year
ORDER BY start_year DESC, rows DESC;


In [ ]:
%sql
-- Uniqueness guard: every funder_award_id (Identifier) must be distinct.
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT funder_award_id) AS distinct_ids
FROM openalex.awards.garfield_weston_raw;


## Step 1.6: Fail-fast - verify Garfield Weston Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row (the UK foundation, funder_id 4320314718).


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320314718;


## Step 2: Transform to award schema

`display_name` = the grant's own `Title`. `start_date` = the real `Award date`. `lead_investigator` is an org-only lead: given/family NULL, `affiliation.name` = recipient org, `affiliation.country` NULL (no recipient-country column; never guessed). `funder_scheme` = Grant Programme. `co_lead_investigator` / `investigators` are NULL.


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.garfield_weston_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320314718  -- Garfield Weston Foundation (UK)
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    COALESCE(n.title, CONCAT('Garfield Weston Foundation grant ', n.funder_award_id)) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN 'GBP' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.grant_programme AS funder_scheme,
    'garfield_weston_foundation' AS provenance,
    TRY_CAST(n.start_date AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.recipient_org IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.recipient_org AS name,
                CAST(NULL AS STRING) AS country,  -- no recipient-country column in source; never guessed
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CAST(NULL AS STRING) AS landing_page_url,  -- 360Giving has no per-grant page
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.garfield_weston_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL;


## Step 3: Insert into openalex_awards_raw at priority 159


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'garfield_weston_foundation' AND priority = 159;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    159 AS priority  -- Garfield Weston Foundation grants
FROM openalex.awards.garfield_weston_awards;


## Step 6: Verification

§6.7 amount-coverage check **applies** (NOT waived): GWF publishes a positive amount on ~99.9% of grants (the `$0`-guard nulls the few blanks). Other checks: display_name / description / start_year / lead (recipient org) ~100%; start_date from the real Award date; currency = [GBP]; lead_investigator.affiliation.country = NULL (no source country field).


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.garfield_weston_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.garfield_weston_awards;


In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_date) AS has_start_date,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.garfield_weston_awards;


In [ ]:
%sql
-- §6.7 amount check (NOT waived). Expect currency = [GBP].
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.garfield_weston_awards;


In [ ]:
%sql
-- Programme (scheme) split.
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year,
       COUNT(amount) AS has_amount,
       ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.garfield_weston_awards
GROUP BY funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
-- Uniqueness + funder struct sanity.
SELECT COUNT(*) AS total, COUNT(DISTINCT id) AS distinct_ids,
       MIN(funder.display_name) AS funder, MIN(funder.doi) AS funder_doi
FROM openalex.awards.garfield_weston_awards;


In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 50) AS title,
       funder_scheme AS programme, funding_type, start_date, start_year, amount, currency,
       lead_investigator.affiliation.name AS org
FROM openalex.awards.garfield_weston_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;
